In [ ]:
s = SparkSession.builder.getOrCreate()

# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
DATABASE_NAME = "nyc_taxi"
TABLE_NAME = "yellow_tripdata"
SOURCE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2026-01.parquet"
LOCAL_FILE = "/tmp/yellow_tripdata_2026_01.parquet"

# -----------------------------------------------------------------------------
# Download source file
# -----------------------------------------------------------------------------
import urllib.request
urllib.request.urlretrieve(SOURCE_URL, LOCAL_FILE)

# -----------------------------------------------------------------------------
# Load dataset
# -----------------------------------------------------------------------------
df = spark.read.parquet(LOCAL_FILE)

# -----------------------------------------------------------------------------
# Create schema/database
# -----------------------------------------------------------------------------
spark.sql(f"""
CREATE DATABASE IF NOT EXISTS {DATABASE_NAME}
COMMENT 'NYC Taxi Trip Record datasets used for analytics demonstrations'
""")

# -----------------------------------------------------------------------------
# Persist as a managed Spark table
# -----------------------------------------------------------------------------
df.write \
    .mode("overwrite") \
    .saveAsTable(f"{DATABASE_NAME}.{TABLE_NAME}")

# -----------------------------------------------------------------------------
# Verify
# -----------------------------------------------------------------------------
spark.sql(f"SELECT COUNT(*) AS total_trips FROM {DATABASE_NAME}.{TABLE_NAME}").show()
spark.sql(f"DESCRIBE {DATABASE_NAME}.{TABLE_NAME}").show(truncate=False)

In [ ]:
%%sql
-- Daily taxi trips
SELECT
    DATE(tpep_pickup_datetime) AS trip_date,
    ROUND(SUM(total_amount), 2) AS total_revenue,
    COUNT(*) AS total_trips
FROM nyc_taxi.yellow_tripdata
GROUP BY DATE(tpep_pickup_datetime)
ORDER BY trip_date;